In [ ]:
import zipfile
import os
import random
import shutil

from torchvision.datasets import Food101
from torch.utils.data import Dataset, Subset

In [ ]:
selected_categories = [
    # --- Asia (13) ---
    'bibimbap', 'gyoza', 'sashimi', 'pad_thai', 'pho', 
    'miso_soup', 'edamame', 'spring_rolls', 'sushi',
    'chicken_curry', 'fried_rice', 'ramen', 'kimchi',
    
    # --- Latino (5) ---
    'tacos', 'enchiladas', 'guacamole', 'ceviche', 'nachos',
    
    # --- Classic Western (11) ---
    'pizza_margherita', 'hamburger', 'hot_dog', 'steak',
    'french_fries', 'grilled_salmon', 'spaghetti_bolognese',
    'lasagna', 'club_sandwich', 'macaroni_and_cheese', 'risotto',
    
    # --- Dessert / Social Media (9) ---
    'tiramisu', 'cheesecake', 'macarons', 'donuts',
    'waffles', 'pancakes', 'ice_cream', 'apple_pie',
    'strawberry_shortcake',
    
    # --- Seafood (2) ---
    'mussels', 'oysters'
    ]

In [ ]:
# 載入 dataset
dataset = Food101(root='./data', split='train', download=True)

In [ ]:
class_to_idx = {cls_name: i for i, cls_name in enumerate(selected_categories)}
idx_to_class = {i: cls_name for cls_name, i in class_to_idx.items()}

In [ ]:
class FilteredFood101(Dataset):
    def __init__(self, dataset, selected_classes, transform=None):
        self.dataset = dataset
        self.transform = transform
        
        self.selected_classes = selected_classes
        self.class_to_idx = {cls: i for i, cls in enumerate(selected_classes)}
        
        self.filtered_indices = []
        
        for i in range(len(dataset)):
            label = dataset._labels[i]
            class_name = dataset.classes[label]
            
            if class_name in self.selected_classes:
                self.filtered_indices.append(i)

    def __len__(self):
        return len(self.filtered_indices)

    def __getitem__(self, idx):
        real_idx = self.filtered_indices[idx]
        img, label = self.dataset[real_idx]
        
        class_name = self.dataset.classes[label]
        new_label = self.class_to_idx[class_name]  # ⭐ 重建label
        
        if self.transform:
            img = self.transform(img)
            
        return img, new_label

In [ ]:
train_dataset = Food101(root='./data', split='train', download=True)
test_dataset = Food101(root='./data', split='test', download=True)

filtered_train = FilteredFood101(train_dataset, selected_categories)
filtered_test = FilteredFood101(test_dataset, selected_categories)

print(len(filtered_train), len(filtered_test))